# DnaN time course: M9glu

Filtered single cells are supplied in Dataset S1. Means are calculated within replicate/time first, then summarized across replicates with SD. Fraction comparisons are inclusive. Morphology QC uses |z|<8 within replicate, C2_NC_ratio in [0.05,1.1], C3_NC_ratio <=5, and excludes the source notebook's specified time points.

In [ ]:
from pathlib import Path
import numpy as np
import numpy.typing
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PACKAGE = Path.cwd()
if not (PACKAGE / "Datasets").is_dir():
    PACKAGE = PACKAGE.parent
DATASETS = PACKAGE / "Datasets"
if not DATASETS.is_dir():
    raise FileNotFoundError("Launch this notebook from the package root or Github folder.")
plt.rcParams.update({"font.family": "Arial", "text.usetex": False})

final_df = pd.read_excel(DATASETS / "Dataset S1.xlsx", sheet_name="M9glu_timecourse_cells").rename(columns={"time_h":"time"})

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from matplotlib.ticker import AutoMinorLocator

def _sigmoid(x, L, x0, k, b):
    with np.errstate(over='ignore'):
        return L / (1 + np.exp(-k * (x - x0))) + b

def plot_features_multipanel(df: pd.DataFrame, features, time_col: str, replicate_col: str, ncols: int=3, width_mm: float=160.0, height_mm: float=80.0, fit_sigmoid: bool=True, x_min: float=0.0, feature_ylims: dict | None=None):
    """
    Multi-panel (up to 6 panels) mean±SD per timepoint across replicates.

    Parameters
    ----------
    x_min
        Left limit of the time axis. Use 0 to start all plots at time = 0 h.
    feature_ylims
        Optional dict with custom y-limits per feature, e.g.
        {"cell_width": (0.7, 1.1)}
    """
    time_colors = {12: '#1c75bc', 14: 'green', 20: 'orange', 24: 'red', 46: '#7f3f98'}
    default_color = '0.35'
    if feature_ylims is None:
        feature_ylims = {}
    features = list(features)[:6]
    n_panels = 6
    plt.rcParams.update({'font.family': 'Arial', 'font.size': 8, 'axes.titlesize': 9, 'axes.labelsize': 8, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5, 'axes.linewidth': 0.5})
    width_in = width_mm / 25.4
    height_in = height_mm / 25.4
    nrows = int(np.ceil(n_panels / ncols))
    (fig, axes) = plt.subplots(nrows=nrows, ncols=ncols, figsize=(width_in, height_in), constrained_layout=True)
    axes = np.array(axes).reshape(-1)
    marker_size = 3.0
    err_linewidth = 0.8
    cap_size = 1.0
    fit_lw = 0.9
    for i in range(n_panels):
        ax = axes[i]
        if i >= len(features):
            ax.axis('off')
            continue
        feature = features[i]
        grouped = df.groupby([time_col, replicate_col])[feature].mean().reset_index()
        grouped[time_col] = pd.to_numeric(grouped[time_col], errors='coerce')
        summary = grouped.groupby(time_col)[feature].agg(['mean', 'std']).reset_index().sort_values(time_col).dropna(subset=[time_col])
        x = summary[time_col].to_numpy(dtype=float)
        y = summary['mean'].to_numpy(dtype=float)
        yerr = summary['std'].to_numpy(dtype=float)
        ax.errorbar(x, y, yerr=yerr, fmt='none', ecolor='black', elinewidth=err_linewidth, capsize=cap_size, capthick=err_linewidth, zorder=2)
        for (xi, yi) in zip(x, y):
            key = int(round(xi))
            c = time_colors.get(key, default_color)
            ax.plot(xi, yi, marker='o', markersize=marker_size, linestyle='none', color=c, markeredgewidth=0, zorder=3)
        if fit_sigmoid and len(np.unique(x)) >= 4:
            p0 = [np.nanmax(y), np.nanmedian(x), 1.0, np.nanmin(y)]
            try:
                (popt, _) = curve_fit(_sigmoid, x, y, p0=p0, maxfev=20000)
                x_fit = np.linspace(np.nanmin(x), np.nanmax(x), 200)
                y_fit = _sigmoid(x_fit, *popt)
                ax.plot(x_fit, y_fit, linestyle='--', linewidth=fit_lw, color='darkgoldenrod', zorder=1)
            except Exception:
                pass
        ax.set_title(feature, fontweight='bold', pad=6)
        ax.set_xlabel('Time (h)')
        ax.set_ylabel(f'Mean {feature}')
        ax.set_box_aspect(0.6)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(which='major', direction='out', length=2.5, width=0.8, labelsize=6.5)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontfamily('Arial')
        if len(x) > 0:
            ax.set_xlim(left=x_min)
        else:
            ax.set_xlim(left=x_min)
        if feature in feature_ylims:
            (ymin, ymax) = feature_ylims[feature]
            ax.set_ylim(ymin, ymax)
    plt.show()
    plt.close(fig)
plot_features_multipanel(final_df, features=['cell_length', 'cell_volume', 'cell_aspect_ratio'], time_col='time', replicate_col='replicate', x_min=0)
plot_features_multipanel(final_df, features=['cell_width', 'cell_area', 'cell_surface_area', 'cell_SOV'], time_col='time', replicate_col='replicate', x_min=0, feature_ylims={'cell_width': (0.8, 1.2)})

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from matplotlib.ticker import MultipleLocator, AutoMinorLocator

def _sigmoid(x, L, x0, k, b):
    with np.errstate(over='ignore'):
        return L / (1 + np.exp(-k * (x - x0))) + b

def plot_fractions_multipanel(df: pd.DataFrame, specs, time_col: str, replicate_col: str, ncols: int=3, width_mm: float=160.0, height_mm: float=80.0, fit_sigmoid: bool=True, x_min: float=0.0):
    """
    Multi-panel (up to 6 panels) FRACTIONS with mean±SD per timepoint across replicates, displayed as figures.
    """
    time_colors = {12: '#1c75bc', 14: 'green', 20: 'orange', 24: 'red', 46: '#7f3f98'}
    default_color = '0.35'
    specs = list(specs)[:6]
    n_panels = 6
    plt.rcParams.update({'font.family': 'Arial', 'font.size': 8, 'axes.titlesize': 9, 'axes.labelsize': 8, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5, 'axes.linewidth': 0.5})
    width_in = width_mm / 25.4
    height_in = height_mm / 25.4
    nrows = int(np.ceil(n_panels / ncols))
    (fig, axes) = plt.subplots(nrows=nrows, ncols=ncols, figsize=(width_in, height_in), constrained_layout=True)
    axes = np.array(axes).reshape(-1)
    marker_size = 3.0
    err_linewidth = 0.8
    cap_size = 1.0
    fit_lw = 0.9
    for i in range(n_panels):
        ax = axes[i]
        if i >= len(specs):
            ax.axis('off')
            continue
        spec = specs[i]
        feature = spec['feature']
        threshold = spec['threshold']
        condition = spec.get('condition', 'above').lower()
        title = spec.get('title', f'{feature} ({condition} {threshold})')
        xlabel = spec.get('xlabel', 'Time (h)')
        ylabel = spec.get('ylabel', 'Fraction')
        ylim = spec.get('ylim', None)
        group_cols = [time_col, replicate_col]
        if condition == 'below':
            target = df[df[feature] <= threshold].groupby(group_cols).size().reset_index(name='target_count')
        else:
            target = df[df[feature] >= threshold].groupby(group_cols).size().reset_index(name='target_count')
        total = df.groupby(group_cols).size().reset_index(name='total_count')
        merged = pd.merge(target, total, on=group_cols, how='right')
        merged['target_count'] = merged['target_count'].fillna(0)
        merged['fraction'] = merged['target_count'] / merged['total_count']
        merged[time_col] = pd.to_numeric(merged[time_col], errors='coerce')
        summary = merged.groupby(time_col)['fraction'].agg(['mean', 'std']).reset_index().sort_values(time_col).dropna(subset=[time_col])
        x = summary[time_col].to_numpy(dtype=float)
        y = summary['mean'].to_numpy(dtype=float)
        yerr = summary['std'].fillna(0).to_numpy(dtype=float)
        ax.errorbar(x, y, yerr=yerr, fmt='none', ecolor='black', elinewidth=err_linewidth, capsize=cap_size, capthick=err_linewidth, zorder=2)
        for (xi, yi) in zip(x, y):
            key = int(round(xi))
            c = time_colors.get(key, default_color)
            ax.plot(xi, yi, marker='o', markersize=marker_size, linestyle='none', color=c, markeredgewidth=0, zorder=3)
        if fit_sigmoid and len(np.unique(x)) >= 4:
            p0 = [np.nanmax(y), np.nanmedian(x), 1.0, np.nanmin(y)]
            try:
                (popt, _) = curve_fit(_sigmoid, x, y, p0=p0, maxfev=20000)
                x_fit = np.linspace(np.nanmin(x), np.nanmax(x), 200)
                y_fit = _sigmoid(x_fit, *popt)
                ax.plot(x_fit, y_fit, linestyle='--', linewidth=fit_lw, color='darkgoldenrod', zorder=1)
            except Exception:
                pass
        ax.set_title(f'{title}\n({condition} {threshold})', fontweight='bold', pad=6)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_box_aspect(0.6)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(which='major', direction='out', length=2.5, width=0.8, labelsize=6.5)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontfamily('Arial')
        if ylim is not None:
            ax.set_ylim(*ylim)
        if len(x) > 0:
            x_max_data = np.nanmax(x)
            x_max_plot = np.ceil(x_max_data / 10) * 10 + 2
            ax.set_xlim(x_min, x_max_plot)
        else:
            ax.set_xlim(x_min, 52)
        ax.xaxis.set_major_locator(MultipleLocator(10))
    plt.show()
    plt.close(fig)
specs = [{'feature': 'cell_constriction_degree_width_axial', 'threshold': 0.1, 'condition': 'above', 'title': 'Fraction of Constricting Cells', 'xlabel': 'Time (h)', 'ylabel': 'Fraction of constricting cells'}, {'feature': 'C2_object_number', 'threshold': 1, 'condition': 'below', 'title': 'Fraction of Cells with One Nucleoid', 'xlabel': 'Time (h)', 'ylabel': 'Fraction of cells with one nucleoid'}, {'feature': 'C3_NC_ratio', 'threshold': 0.4, 'condition': 'below', 'title': 'Fraction of Cells with DnaN foci', 'xlabel': 'Time (h)', 'ylabel': 'Fraction of cells with DnaN foci'}]
plot_fractions_multipanel(final_df, specs=specs, time_col='time', replicate_col='replicate', width_mm=160, height_mm=80, ncols=3, x_min=0)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

def plot_ratio_with_errorbars(df, feature1, feature2, time_col, replicate_col, color='blue', title=None, xlabel=None, ylabel=None):
    """
    Plots the ratio of the mean of two features per time point with error bars 
    (error propagated from std of both features). 
    Also computes correlation between feature means at each timepoint.

    Parameters:
    df (pd.DataFrame): The dataframe containing the data.
    feature1 (str): First feature/column.
    feature2 (str): Second feature/column.
    time_col (str): The column indicating the time points.
    replicate_col (str): The column indicating the replicates.
    color (str): Color of the plot.
    title (str, optional): Title of the plot. If None, a default title is used.
    xlabel (str, optional): X-axis label.
    ylabel (str, optional): Y-axis label.
    """
    grouped1 = df.groupby([time_col, replicate_col])[feature1].mean().reset_index()
    summary1 = grouped1.groupby(time_col)[feature1].agg(['mean', 'std']).reset_index()
    grouped2 = df.groupby([time_col, replicate_col])[feature2].mean().reset_index()
    summary2 = grouped2.groupby(time_col)[feature2].agg(['mean', 'std']).reset_index()
    summary = pd.merge(summary1, summary2, on=time_col, suffixes=(f'_{feature1}', f'_{feature2}'))
    mean_ratio = summary[f'mean_{feature1}'] / summary[f'mean_{feature2}']
    error_ratio = mean_ratio * np.sqrt((summary[f'std_{feature1}'] / summary[f'mean_{feature1}']) ** 2 + (summary[f'std_{feature2}'] / summary[f'mean_{feature2}']) ** 2)
    plt.figure(figsize=(5, 3))
    ax = plt.gca()
    ax.errorbar(summary[time_col], mean_ratio, yerr=error_ratio, fmt='o', color=color, ecolor='black', elinewidth=0.5, markersize=4, capsize=1)
    ax.set_xlabel(xlabel if xlabel else 'Time (h)', font='Arial', fontsize=12)
    ax.set_ylabel(ylabel if ylabel else f'Ratio {feature1}/{feature2}', font='Arial', fontsize=12)
    ax.set_title(title if title else f'Ratio of {feature1}/{feature2} per Time Point', font='Arial', fontsize=16, fontweight='bold', y=1.1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    (rho, pval) = spearmanr(summary[f'mean_{feature1}'], summary[f'mean_{feature2}'])
    ax.text(0.95, 0.05, f'ρ = {rho:.3f}\np = {pval:.3g}', transform=ax.transAxes, fontsize=10, ha='right', va='bottom', color='darkred')
    plt.show()

## PHATE
The figure uses the supplied coordinates. The optional branch recalculates PHATE from the supplied sampled feature matrix (seed 42; 10 neighbors; standardized features). Constriction rescaling is for display only.

In [ ]:
df_subset = pd.read_excel(DATASETS / "Dataset S1.xlsx", sheet_name="M9glu_PHATE_plotted").rename(columns={"time_h":"time"})
RECOMPUTE_PHATE = False
if RECOMPUTE_PHATE:
    import phate
    from sklearn.preprocessing import StandardScaler
    inputs = pd.read_excel(DATASETS / "Dataset S1.xlsx", sheet_name="M9glu_PHATE_inputs")
    features = [c for c in inputs if c not in {"sample_id", "cell_id"}]
    X = StandardScaler().fit_transform(inputs[features])
    df_subset[["PHATE1", "PHATE2"]] = phate.PHATE(knn=10, random_state=42, n_jobs=1).fit_transform(X)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
df_subset_filtered = df_subset.copy()
time_order = sorted(df_subset_filtered['time'].unique())
highlight_colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728', '#9467bd']
(fig, axes) = plt.subplots(1, 5, figsize=(20, 4), dpi=800)
axes = axes.flatten()
for (idx, time_highlight) in enumerate(time_order):
    ax = axes[idx]
    highlight_color = highlight_colors[idx]
    mask_highlight = df_subset_filtered['time'] == time_highlight
    mask_other = df_subset_filtered['time'] != time_highlight
    ax.scatter(df_subset_filtered[mask_other]['PHATE1'], df_subset_filtered[mask_other]['PHATE2'], c='#C0C0C0', s=4, alpha=0.4, edgecolors='none', rasterized=True)
    ax.scatter(df_subset_filtered[mask_highlight]['PHATE1'], df_subset_filtered[mask_highlight]['PHATE2'], c=highlight_color, s=4, alpha=0.8, edgecolors='none', rasterized=True)
    legend_elements = [Patch(facecolor='#C0C0C0', label='Other'), Patch(facecolor=highlight_color, label=f'{time_highlight} h')]
    ax.legend(handles=legend_elements, title='Time (h)', title_fontsize=7, fontsize=6, loc='best', frameon=False)
    ax.set_title(f'Timepoint: {time_highlight} h', fontsize=9, pad=8)
    ax.set_xlabel('PHATE 1', fontsize=8)
    ax.set_ylabel('PHATE 2', fontsize=8)
    sns.despine(ax=ax)
fig.suptitle('PHATE Embedding - Individual Timepoint Highlights', fontsize=12, y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
df_subset_filtered = df_subset[df_subset['time'] != 9].copy()
df_subset_filtered['cell_constriction_degree_width_axial_processed'] = df_subset_filtered['cell_constriction_degree_width_axial'].copy()
df_subset_filtered.loc[df_subset_filtered['cell_constriction_degree_width_axial_processed'] > 0.5, 'cell_constriction_degree_width_axial_processed'] = np.nan
df_subset_filtered['cell_constriction_degree_width_axial_processed'] *= 2
original_cmap = plt.cm.YlOrBr_r
colors = original_cmap(np.linspace(0.0, 0.7, 100))
custom_cmap = LinearSegmentedColormap.from_list('YlOrBr_trimmed', colors)
features_to_plot = ['cell_volume', 'cell_aspect_ratio', 'cell_constriction_degree_width_axial_processed', 'C2_object_number', 'C2_NC_ratio']
feature_labels = ['Cell Volume', 'Cell Aspect Ratio', 'Constriction degree', 'Nucleoid number', 'NC ratio']
(fig, axes) = plt.subplots(1, 5, figsize=(20, 5), dpi=300)
axes = axes.flatten()
scatters = []
for (feat_idx, (feature, feature_label)) in enumerate(zip(features_to_plot, feature_labels)):
    ax = axes[feat_idx]
    mask_valid = ~df_subset_filtered[feature].isna()
    scatter = ax.scatter(df_subset_filtered.loc[mask_valid, 'PHATE1'], df_subset_filtered.loc[mask_valid, 'PHATE2'], c=df_subset_filtered.loc[mask_valid, feature], s=4, alpha=0.7, edgecolors='none', cmap=custom_cmap, rasterized=True)
    scatters.append((scatter, feature_label))
    ax.set_title(feature_label, fontsize=9, pad=8)
    ax.set_xlabel('PHATE 1', fontsize=8)
    ax.set_ylabel('PHATE 2', fontsize=8)
    ax.tick_params(labelsize=7)
    sns.despine(ax=ax)
fig.suptitle('PHATE Embedding - Feature Distribution (All Timepoints)', fontsize=12, y=0.995)
plt.tight_layout(rect=[0, 0.14, 1, 0.95])
for (ax, (scatter, label)) in zip(axes, scatters):
    pos = ax.get_position()
    cbar_height = 0.018
    cbar_gap = 0.055
    cax = fig.add_axes([pos.x0, pos.y0 - cbar_gap, pos.width, cbar_height])
    cbar = fig.colorbar(scatter, cax=cax, orientation='horizontal')
    cbar.set_label(label, fontsize=7, labelpad=2)
    cbar.ax.tick_params(labelsize=6, pad=1)
plt.show()